In [1]:
# Install the HuggingFace transformers library
!pip install transformers torch pandas tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import torch
import os
from transformers import RobertaTokenizer, RobertaModel
from torch.nn.functional import cosine_similarity
from tqdm.notebook import tqdm

DRIVE_DIR = "/content/drive/MyDrive/Thesis_Project/data/processed/"

FILES =[
    "01_Original_12k.csv",
    "02_Distractor_A_Swapped_12k.csv",
    "03_Distractor_B_Shuffled_12k.csv"
]

# Ensure GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware: {device}")

# Load CodeBERT Base
print("Loading CodeBERT model and tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
model = RobertaModel.from_pretrained("microsoft/codebert-base")
model.to(device)
model.eval() # Set model to inference mode to save memory

print("Model loaded successfully.")

Using hardware: cuda
Loading CodeBERT model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Model loaded successfully.


In [3]:
def get_codebert_similarity(code, comment):
    # Handle any potential null values
    code = str(code) if pd.notna(code) else ""
    comment = str(comment) if pd.notna(comment) else ""

    # Tokenize the inputs and truncate to 512 tokens
    code_tokens = tokenizer(code, return_tensors='pt', truncation=True, max_length=512).to(device)
    comment_tokens = tokenizer(comment, return_tensors='pt', truncation=True, max_length=512).to(device)

    with torch.no_grad(): # Disable gradient calculation for faster inference
        code_output = model(**code_tokens)
        comment_output = model(**comment_tokens)

        # Extract the [CLS] token representation (index 0) which holds the sequence meaning
        code_embedding = code_output.last_hidden_state[:, 0, :]
        comment_embedding = comment_output.last_hidden_state[:, 0, :]

        # Calculate Cosine Similarity
        similarity = cosine_similarity(code_embedding, comment_embedding)

    return similarity.item()

In [4]:
BATCH_SIZE = 500

for file_name in FILES:
    input_path = os.path.join(DRIVE_DIR, file_name)
    output_name = file_name.replace(".csv", "_scored.csv")
    output_path = os.path.join(DRIVE_DIR, output_name)

    # Load dataset. Resume from output file if it already exists to prevent data loss.
    if os.path.exists(output_path):
        print(f"Resuming from existing file: {output_name}")
        df = pd.read_csv(output_path)
    else:
        print(f"Starting fresh for: {file_name}")
        df = pd.read_csv(input_path)
        if 'codebert_score' not in df.columns:
            df['codebert_score'] = pd.NA

    # Find rows that still need to be processed
    missing_indices = df[df['codebert_score'].isna()].index.tolist()

    if not missing_indices:
        print(f"{file_name} is completely scored.")
        continue

    print(f"Rows remaining for {file_name}: {len(missing_indices)}")

    # Process the missing rows in batches
    for i in range(0, len(missing_indices), BATCH_SIZE):
        chunk_indices = missing_indices[i:i + BATCH_SIZE]
        print(f"Processing batch {i} to {i + len(chunk_indices)}...")

        for idx in tqdm(chunk_indices, leave=False):
            code = df.at[idx, 'original_code']
            comment = df.at[idx, 'comment']

            # Calculate and assign score
            score = get_codebert_similarity(code, comment)
            df.at[idx, 'codebert_score'] = score

        # Save dataframe to Google Drive after every batch completes
        df.to_csv(output_path, index=False)
        print(f"Batch saved to {output_name}.")

    print(f"Completed scoring for {file_name}.\n")

Starting fresh for: 01_Original_12k.csv
Rows remaining for 01_Original_12k.csv: 12000
Processing batch 0 to 500...


  0%|          | 0/500 [00:00<?, ?it/s]

Batch saved to 01_Original_12k_scored.csv.
Processing batch 500 to 1000...


  0%|          | 0/500 [00:00<?, ?it/s]

Batch saved to 01_Original_12k_scored.csv.
Processing batch 1000 to 1500...


...

Batch saved to 03_Distractor_B_Shuffled_12k_scored.csv.
Processing batch 11000 to 11500...


  0%|          | 0/500 [00:00<?, ?it/s]

Batch saved to 03_Distractor_B_Shuffled_12k_scored.csv.
Processing batch 11500 to 12000...


  0%|          | 0/500 [00:00<?, ?it/s]

Batch saved to 03_Distractor_B_Shuffled_12k_scored.csv.
Completed scoring for 03_Distractor_B_Shuffled_12k.csv.

